# TESSERA embeddings — access

**Purpose.** Establish, once, how to read TESSERA foundation-model
embeddings for the Coimbra district, so other notebooks (segmentation
mask, RQ1) don't each re-figure it out. Nothing is modelled here.

**Established fact.** TESSERA (Feng et al., 2025) gives a 128-dim
embedding per 10 m pixel from Sentinel-1/2 time series, MIT licence;
see [`docs/methodology.md`](../../docs/methodology.md#foundation-models).

**What was tried before:** the deleted RQ1 pilot (see
[`data/README.md`](../../data/README.md)) already read this same
store, with `GeoTesseraZarr.sample_points`, one embedding per
observation pixel (~29 000 points, ~2 h). This notebook instead reads
a small **patch** around each point (`read_patch`), needed for region
growing, not just the one pixel.

**Access (confirmed working here):** `geotessera` 0.11.0,
`geotessera.store.GeoTesseraZarr`, dataset variant `dclimate` (the
default for v1.1), reading directly over HTTPS from
`https://data.source.coop/tessera/tessera/zarr/v1.1-dclimate`
(public, no credentials). No bulk download needed: reads are
per-point/per-patch/per-bbox, with a local byte-range cache in
`data/raw/satellite/tessera_zarr_cache`. A first read near a new area
takes a few seconds; nearby points reuse cached chunks (~1 s).

*Earlier in this session I wrongly concluded this was blocked,
testing only the high-level `GeoTessera` class, which only reads the
`cambridge` NPY dataset (no Portugal coverage) — `GeoTesseraZarr` is
a separate, lower-level class that reads the right (`dclimate`)
dataset directly.*

> **Question numbering (note of 2026-11-13).** Current numbering in
> `docs/research_questions.md`: RQ1 = effort-reduction curve (citizen
> science against expert data); RQ2 = from points to reference data
> (segmentation mask, grade and accuracy filters); RQ3 =
> transferability away from the sampled area; RQ4 = TESSERA against
> Sentinel-2.
> In this notebook "RQ1 pilot" is the name of the stored embedding set
> (see `data/README.md`), not a question number. The segmentation mask is RQ2.

## 1. Read one patch

A 21×21 pixel patch (210×210 m) around one *A. dealbata* presence, at
the native 10 m grid, in its native UTM zone (29N — same CRS as
`src/grid.py`'s pixel ids).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from geotessera.store import GeoTesseraZarr

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CACHE_DIR = ROOT / "data" / "raw" / "satellite" / "tessera_zarr_cache"
YEAR = 2023

store = GeoTesseraZarr(cache_dir=CACHE_DIR)

acacia = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
lon0, lat0 = acacia.loc[0, ["longitude", "latitude"]]
patch, transform, crs = store.read_patch(lon=lon0, lat=lat0, year=YEAR, size_px=21)
print(f"patch {patch.shape}, dtype {patch.dtype}, crs {crs}")
print(f"NaN pixels: {np.isnan(patch).any(axis=-1).sum()}/{patch.shape[0] * patch.shape[1]}")

## 2. Sanity check: does it look like a real place?

128 bands aren't directly visible; reduce to 3 with PCA (fit on this
patch only, just to look at it) and plot as RGB, with the presence
pixel marked.

In [ ]:
from sklearn.decomposition import PCA

flat = patch.reshape(-1, patch.shape[-1])
rgb = PCA(n_components=3).fit_transform(flat).reshape(*patch.shape[:2], 3)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(rgb)
ax.scatter([10], [10], c="red", marker="x", s=80, label="presence pixel")
ax.set_title(f"TESSERA {YEAR}, PCA-RGB, 210×210 m around one presence")
ax.legend()
plt.show()